# AutoMorphoTrack Full Source Code
This notebook contains the complete source code of the AutoMorphoTrack package.

## `Lyso_count.py`

In [ ]:
import numpy as np
import cv2
from skimage.measure import label, regionprops
import matplotlib.pyplot as plt


def count_lysosomes(binary_masks, min_area=1):
    """
    Counts lysosomes in each frame based on binary masks.

    Parameters:
        binary_masks (np.ndarray): 3D array of binary masks (frames x height x width).
        min_area (float): Minimum area (in pixels) to count an object as a lysosome.

    Returns:
        counts_per_frame (list): Lysosome count per frame.
        labeled_masks (list): List of label images per frame.
    """
    counts_per_frame = []
    labeled_masks = []

    for frame in binary_masks:
        labeled = label(frame)
        props = regionprops(labeled)
        valid_regions = [p for p in props if p.area >= min_area]
        count = len(valid_regions)
        counts_per_frame.append(count)
        labeled_frame = np.zeros_like(frame)
        for i, region in enumerate(valid_regions, start=1):
            labeled_frame[labeled == region.label] = i
        labeled_masks.append(labeled_frame)

    return counts_per_frame, labeled_masks


def visualize_lysosome_counts(image_stack, labeled_masks, counts, save_path=None):
    """
    Overlay lysosome counts and labels on the first frame of the stack.

    Parameters:
        image_stack (np.ndarray): Original grayscale or RGB stack.
        labeled_masks (list): List of labeled masks (one per frame).
        counts (list): List of counts per frame.
        save_path (str): Optional path to save the visualization.
    """
    frame = image_stack[0]
    labeled = labeled_masks[0]
    plt.figure(figsize=(8, 8))
    plt.imshow(frame, cmap='gray')
    props = regionprops(labeled)

    for region in props:
        y, x = region.centroid
        plt.text(x, y, f"{region.label}", color='lime', fontsize=10, ha='center', va='center')

    plt.title(f"Lysosomal Count (Frame 0): {counts[0]}", fontsize=14)
    plt.axis('off')

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


## `__init__.py`

In [ ]:
from .core import run_full_pipeline


## `core.py`

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk

from .detection import detect_organelles
from .tracking import calculate_displacement
from .morphology import classify_mitochondria
from .visualization import plot_organelles

def run_full_pipeline(image_stack):
    for frame in image_stack:
        mask = detect_organelles(frame)
        plot_organelles(frame, mask)


## `detection.py`

In [ ]:
from skimage.morphology import dilation, disk
import numpy as np
import cv2

def detect_organelles(image, method='adaptive'):
    if method == 'adaptive':
        thresh = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 11, 2)
    else:
        _, thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresh

def generate_frame0_detection_overlay(lyso_frame, mito_frame, output_path="organelle_detection_overlay_frame0.png"):
    from skimage.filters import threshold_otsu
    from scipy.ndimage import gaussian_filter
    import numpy as np
    import cv2
    import os

    # Preprocess lysosome frame
    lyso_blur = gaussian_filter(lyso_frame, sigma=1)
    lyso_thresh = threshold_otsu(lyso_blur)
    lyso_mask = lyso_blur > lyso_thresh
    lyso_overlay = lyso_frame.copy()
    lyso_overlay[lyso_mask] = 1.0

    # Preprocess mitochondria frame
    mito_blur = gaussian_filter(mito_frame, sigma=1)
    mito_thresh = threshold_otsu(mito_blur)
    mito_mask = mito_blur > mito_thresh
    mito_overlay = mito_frame.copy()
    mito_overlay[mito_mask] = 1.0

    # Convert to 8-bit
    lyso_img_uint8 = (lyso_overlay * 255).astype(np.uint8)
    mito_img_uint8 = (mito_overlay * 255).astype(np.uint8)

    # Concatenate (lysosome left, mitochondria right)
    combined = np.concatenate((lyso_img_uint8, mito_img_uint8), axis=1)

    # Save output
    cv2.imwrite(output_path, combined)

def generate_frame0_contour_overlay(lyso_frame, mito_frame, output_path="organelle_detection_overlay_frame0_contours.png"):
    from skimage.filters import threshold_otsu
    from scipy.ndimage import gaussian_filter
    from skimage.measure import find_contours
    import numpy as np
    import cv2

    lyso_blur = gaussian_filter(lyso_frame, sigma=1)
    lyso_thresh = threshold_otsu(lyso_blur)
    lyso_mask = lyso_blur > lyso_thresh
    lyso_img = (lyso_frame * 255).astype(np.uint8)
    lyso_img_rgb = cv2.cvtColor(lyso_img, cv2.COLOR_GRAY2RGB)

    mito_blur = gaussian_filter(mito_frame, sigma=1)
    mito_thresh = threshold_otsu(mito_blur)
    mito_mask = mito_blur > mito_thresh
    mito_img = (mito_frame * 255).astype(np.uint8)
    mito_img_rgb = cv2.cvtColor(mito_img, cv2.COLOR_GRAY2RGB)

    for contour in find_contours(lyso_mask, 0.5):
        contour = np.round(contour).astype(np.int32)
        for y, x in contour:
            cv2.circle(lyso_img_rgb, (x, y), 1, (255, 0, 0), -1)

    for contour in find_contours(mito_mask, 0.5):
        contour = np.round(contour).astype(np.int32)
        for y, x in contour:
            cv2.circle(mito_img_rgb, (x, y), 1, (255, 0, 0), -1)

    # Mitochondria on left, Lysosomes on right
    combined = np.concatenate((mito_img_rgb, lyso_img_rgb), axis=1)
    cv2.imwrite(output_path, combined)

## `morphology.py`

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi

def classify_mitochondria(properties):
    classified = []
    for prop in properties:
        if prop.area >= 0.025 and prop.eccentricity >= 0.9:
            classified.append('elongated')
        elif prop.area >= 0.02:
            classified.append('punctate')
    return classified



import matplotlib.pyplot as plt
from skimage.measure import label, regionprops
import numpy as np

def save_mito_morphology_image(first_frame, labeled_mask, morphology_labels, output_path="morphology_labeled_frame.png"):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(first_frame, cmap='gray')
    props = regionprops(labeled_mask)

    for i, prop in enumerate(props):
        y, x = prop.centroid
        label_text = morphology_labels.get(prop.label, "")
        ax.text(x, y, label_text, color='red', fontsize=12, ha='center', va='center')

    ax.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()


## `test_basic.py`

In [ ]:

def test_dummy():
    assert True


## `tracking.py`

In [ ]:

import numpy as np

def calculate_displacement(tracks):
    displacement = [np.linalg.norm(track[-1] - track[0]) for track in tracks]
    return displacement


## `visualization.py`

In [ ]:
import matplotlib.pyplot as plt

def plot_organelles(image, mask):
    plt.imshow(image, cmap='gray')
    plt.contour(mask, colors='red')
    plt.show()


import matplotlib.pyplot as plt
import numpy as np

def visualize_colocalization_frame0(mito_stack, lyso_stack):
    frame0_mito = mito_stack[0]
    frame0_lyso = lyso_stack[0]

    # Normalize images for display
    norm_mito = frame0_mito / np.max(frame0_mito)
    norm_lyso = frame0_lyso / np.max(frame0_lyso)

    # Colocalized pixels where both mito and lyso intensities are non-zero
    colocalized = np.logical_and(norm_mito > 0.1, norm_lyso > 0.1)

    rgb = np.zeros((*frame0_mito.shape, 3))
    rgb[..., 1] = norm_mito  # Red channel
    rgb[..., 0] = norm_lyso  # Green channel
    rgb[..., 2] = colocalized.astype(float)  # Cyan = Blue + Green, but we'll mark it with Blue

    plt.figure(figsize=(6, 6))
    plt.imshow(rgb)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig("frame0_colocalization_cyan.png", dpi=300, bbox_inches='tight')
    plt.close()


import matplotlib.pyplot as plt
import numpy as np

def draw_mitochondrial_morphology(frame_mask, props, filename="mitochondria_morphology_only.png"):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(frame_mask, cmap='Reds')
    for prop in props:
        y, x = prop.centroid
        if prop.area >= 0.2 and prop.eccentricity >= 0.85:
            ax.text(x, y, 'E', color='red', fontsize=12, ha='center', va='center')
        elif prop.area >= 0.03 and prop.eccentricity <= 0.7:
            ax.text(x, y, 'P', color='blue', fontsize=12, ha='center', va='center')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

def draw_lysosome_count(frame_mask, props, filename="lysosome_count_only.png"):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(frame_mask, cmap='Greens')
    for i, prop in enumerate(props, 1):
        y, x = prop.centroid
        ax.text(x, y, str(i), color='lime', fontsize=8, ha='center', va='center')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
